# PR-6 — Merging & Validation

**Purpose**: Consolidate the five cleaned interim datasets into a single analytical fact table at engagement-event grain, per PRD Section 5. This notebook covers PR6 (Multi-Source Merging & Join Validation).

**Inputs**:

- data/interim/members_master_clean.csv
- data/interim/subscription_renewal_records_clean.csv
- data/interim/streak_history_episodes_clean.csv
- data/interim/gym_checkin_workout_logs_clean.csv
- data/interim/app_engagement_events_clean.csv
  
**Outputs**:

- data/interim/engagement_events_unified.csv (union of checkin + app, pre-enrichment)
- data/processed/streakforge_merged.csv (fully enriched analytical table, ~206K rows)
- data/interim/pr6_merge_validation_report.csv (join/merge validation log)

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

INTERIM_DIR = Path("../data/interim")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

AS_OF_DATE = pd.Timestamp("2026-08-14")   # same observation cutoff used in notebook 01
GRACE_PERIOD_DAYS = 30                    # matches spec's 30-day lapse/grace window

# --- load all five cleaned sources, re-parsing date columns (CSV round-trip loses dtype) ---
members = pd.read_csv(
    INTERIM_DIR / "members_master_clean.csv",
    parse_dates=["date_of_birth", "join_date"]
)

subs = pd.read_csv(
    INTERIM_DIR / "subscription_renewal_records_clean.csv",
    parse_dates=["billing_cycle_start", "billing_cycle_end"]
)

streak = pd.read_csv(
    INTERIM_DIR / "streak_history_episodes_clean.csv",
    parse_dates=["streak_start_date", "streak_end_date"]
)

checkin = pd.read_csv(
    INTERIM_DIR / "gym_checkin_workout_logs_clean.csv",
    parse_dates=["checkin_datetime", "checkout_datetime"]
)

app = pd.read_csv(
    INTERIM_DIR / "app_engagement_events_clean.csv",
    parse_dates=["event_datetime"]
)

print(f"members : {members.shape}")
print(f"subs    : {subs.shape}")
print(f"streak  : {streak.shape}")
print(f"checkin : {checkin.shape}")
print(f"app     : {app.shape}")

members : (17500, 20)
subs    : (45088, 14)
streak  : (40177, 11)
checkin : (108744, 14)
app     : (97988, 10)


In [3]:

merge_validation_log = []

def log_merge_check(check_name, dataset, passed_count, total_count, note=""):
    """Records a merge/join validation check. Mirrors the PR5 log_rule() pattern
    so PR6's report has the same shape as pr5_validation_report.csv."""
    failed_count = total_count - passed_count
    pct_failed = round((failed_count / total_count) * 100, 2) if total_count else 0.0
    merge_validation_log.append({
        "check": check_name,
        "dataset": dataset,
        "failed_or_flagged": failed_count,
        "total_rows": total_count,
        "pct_flagged": pct_failed,
        "note": note
    })
    print(f"[{check_name}] {dataset}: {failed_count}/{total_count} flagged ({pct_failed}%) — {note}")

In [4]:
# --- common spine columns, renamed to a unified name ---
checkin_common = checkin.rename(columns={
    "checkin_id": "source_event_id",
    "checkin_datetime": "event_datetime",
    "day_of_week": "event_day_of_week",
    "checkin_hour": "event_hour",
}).assign(
    event_source="Gym Checkin",
    event_type=checkin["workout_type"],                       # what the event *was*
    duration_seconds=(
        checkin["actual_duration_minutes"].fillna(checkin["duration_minutes"]) * 60
    ),
    is_duration_outlier_flag=checkin["is_duration_outlier"],
)

app_common = app.rename(columns={
    "event_id": "source_event_id",
    "event_day_of_week": "event_day_of_week",
    "event_hour": "event_hour",
}).assign(
    event_source="App Event",
    duration_seconds=app["session_duration_seconds"],
    is_duration_outlier_flag=app["is_session_duration_outlier"],
)

COMMON_COLS = [
    "source_event_id", "member_id", "event_datetime", "event_source",
    "event_type", "event_day_of_week", "event_hour",
    "duration_seconds", "is_duration_outlier_flag",
]

# checkin-only extension columns (NaN for app rows — realistic sparsity, same
# pattern as trainer_id/notification_engagement nulls handled in PR2)
CHECKIN_ONLY = ["branch_id", "trainer_id", "is_checkout_missing", "season_tag"]
# app-only extension columns (NaN for checkin rows)
APP_ONLY = ["platform", "notification_engagement"]

engagement_events = pd.concat(
    [
        checkin_common[COMMON_COLS + CHECKIN_ONLY],
        app_common[COMMON_COLS + APP_ONLY],
    ],
    ignore_index=True,
    sort=False,
)

# surrogate PK — guarantees uniqueness independent of source ID quality (see next cell)
engagement_events.insert(0, "engagement_event_id", [f"EVT-{100000+i}" for i in range(len(engagement_events))])

print(f"engagement_events (unioned): {engagement_events.shape}")
assert len(engagement_events) == len(checkin) + len(app), "Union row count mismatch — investigate before proceeding"

engagement_events (unioned): (206732, 16)


In [5]:
log_merge_check(
    "union_row_count_conservation",
    "engagement_events",
    passed_count=len(engagement_events),
    total_count=len(checkin) + len(app),
    note="rows after union must exactly equal checkin + app row counts (no gain/loss)"
)

# PR3 checked business-key duplicates on [member_id, event_datetime, event_type] for `app`,
# but never checked source_event_id uniqueness directly — check it now, since it feeds any
# future join back to the raw source files.
dupe_source_ids = engagement_events[
    engagement_events.duplicated(subset="source_event_id", keep=False)
].sort_values("source_event_id")

log_merge_check(
    "source_event_id_uniqueness",
    "engagement_events",
    passed_count=len(engagement_events) - len(dupe_source_ids),
    total_count=len(engagement_events),
    note="raw source_event_id not guaranteed unique across/within sources — surrogate engagement_event_id used as the real PK"
)

print(f"\n{len(dupe_source_ids)} rows share a source_event_id with another row (different member/timestamp):")
print(dupe_source_ids[["engagement_event_id", "source_event_id", "member_id", "event_datetime", "event_source"]])


[union_row_count_conservation] engagement_events: 0/206732 flagged (0.0%) — rows after union must exactly equal checkin + app row counts (no gain/loss)
[source_event_id_uniqueness] engagement_events: 10/206732 flagged (0.0%) — raw source_event_id not guaranteed unique across/within sources — surrogate engagement_event_id used as the real PK

10 rows share a source_event_id with another row (different member/timestamp):
       engagement_event_id source_event_id   member_id      event_datetime  \
109995          EVT-209995      APP-010954  MBR-014492 2025-02-22 05:24:21   
138057          EVT-238057      APP-010954  MBR-012886 2024-05-12 10:23:58   
114147          EVT-214147      APP-018833  MBR-006893 2022-05-17 18:06:43   
130930          EVT-230930      APP-018833  MBR-012444 2024-08-03 20:38:26   
133044          EVT-233044      APP-045157  MBR-015302 2025-09-17 05:14:40   
157512          EVT-257512      APP-045157  MBR-005347 2025-05-21 05:12:12   
143086          EVT-243086     

In [6]:
def validate_referential_integrity(fact_df, fact_name, key="member_id"):
    orphans = fact_df[~fact_df[key].isin(members["member_id"])]
    log_merge_check(
        "referential_integrity_member_id",
        fact_name,
        passed_count=len(fact_df) - len(orphans),
        total_count=len(fact_df),
        note="rows whose member_id has no matching members_master record"
    )
    return orphans

_ = validate_referential_integrity(engagement_events, "engagement_events")
_ = validate_referential_integrity(subs, "subscription_renewal_records")
_ = validate_referential_integrity(streak, "streak_history_episodes")

[referential_integrity_member_id] engagement_events: 0/206732 flagged (0.0%) — rows whose member_id has no matching members_master record
[referential_integrity_member_id] subscription_renewal_records: 0/45088 flagged (0.0%) — rows whose member_id has no matching members_master record
[referential_integrity_member_id] streak_history_episodes: 0/40177 flagged (0.0%) — rows whose member_id has no matching members_master record


In [7]:
pre_merge_rows = len(engagement_events)

merged = engagement_events.merge(
    members,
    on="member_id",
    how="left",
    validate="m:1"          # hard guarantee: this join cannot fan out engagement_events
)

log_merge_check(
    "members_master_join_row_conservation",
    "merged",
    passed_count=len(merged),
    total_count=pre_merge_rows,
    note="left join on members_master must not change row count (validate='m:1' also enforces this)"
)

[members_master_join_row_conservation] merged: 0/206732 flagged (0.0%) — left join on members_master must not change row count (validate='m:1' also enforces this)


In [8]:
merged_sorted = merged.sort_values("event_datetime").reset_index(drop=True)
subs_sorted = subs.sort_values("billing_cycle_start").reset_index(drop=True)

merged = pd.merge_asof(
    merged_sorted,
    subs_sorted,
    left_on="event_datetime",
    right_on="billing_cycle_start",
    by="member_id",
    direction="backward",         # most recent subscription that started on/before the event
    suffixes=("", "_sub"),
)

# merge_asof always returns exactly one row per left row (or NaN) — verify no fan-out occurred
log_merge_check(
    "subscriptions_asof_join_row_conservation",
    "merged",
    passed_count=len(merged),
    total_count=pre_merge_rows,
    note="merge_asof is fan-out safe by construction — row count must equal engagement_events"
)

# --- derive subscription_status_at_event, since a matched "nearest prior" cycle
#     may have already ended by the event date (lapsed/grace-period patterns) ---
has_prior_sub = merged["billing_cycle_start"].notna()
days_past_end = (merged["event_datetime"] - merged["billing_cycle_end"]).dt.days

conditions = [
    ~has_prior_sub,
    merged["event_datetime"] <= merged["billing_cycle_end"],
    (days_past_end > 0) & (days_past_end <= GRACE_PERIOD_DAYS),
    days_past_end > GRACE_PERIOD_DAYS,
]
choices = ["No Prior Subscription", "Active", "Grace Period", "Lapsed"]
merged["subscription_status_at_event"] = np.select(conditions, choices, default="Unknown")

status_counts = merged["subscription_status_at_event"].value_counts()
print(status_counts)
print((status_counts / len(merged) * 100).round(2))

log_merge_check(
    "subscription_status_distribution",
    "merged",
    passed_count=(merged["subscription_status_at_event"] == "Active").sum(),
    total_count=len(merged),
    note=f"only {(merged['subscription_status_at_event']=='Active').mean()*100:.1f}% of events fall inside an active "
         f"billing cycle — expected per design philosophy (silent/soft churn, late renewals), not a join defect"
)


[subscriptions_asof_join_row_conservation] merged: 0/206732 flagged (0.0%) — merge_asof is fan-out safe by construction — row count must equal engagement_events
subscription_status_at_event
Lapsed          147221
Active           51400
Grace Period      7720
Unknown            391
Name: count, dtype: int64
subscription_status_at_event
Lapsed          71.21
Active          24.86
Grace Period     3.73
Unknown          0.19
Name: count, dtype: float64
[subscription_status_distribution] merged: 155332/206732 flagged (75.14%) — only 24.9% of events fall inside an active billing cycle — expected per design philosophy (silent/soft churn, late renewals), not a join defect


In [9]:
streak_sorted = streak.sort_values("streak_start_date").reset_index(drop=True)
merged_sorted2 = merged.sort_values("event_datetime").reset_index(drop=True)

merged = pd.merge_asof(
    merged_sorted2,
    streak_sorted,
    left_on="event_datetime",
    right_on="streak_start_date",
    by="member_id",
    direction="backward",
    suffixes=("", "_streak"),
)

log_merge_check(
    "streaks_asof_join_row_conservation",
    "merged",
    passed_count=len(merged),
    total_count=pre_merge_rows,
    note="merge_asof is fan-out safe by construction — row count must equal engagement_events"
)

# ongoing streaks have streak_end_date = NaT (kept from PR2) — treat as open-ended,
# so "event <= end" only applies when the streak has actually closed
has_prior_streak = merged["streak_start_date"].notna()
still_open = merged["streak_end_date"].isna()
within_closed_streak = merged["event_datetime"] <= merged["streak_end_date"]

merged["is_within_active_streak"] = np.where(
    has_prior_streak & (still_open | within_closed_streak),
    True,
    False,
)

log_merge_check(
    "streak_match_rate",
    "merged",
    passed_count=merged["is_within_active_streak"].sum(),
    total_count=len(merged),
    note="share of events that fall inside a known streak episode for that member"
)

# known data-quality caveat carried forward from earlier profiling: some members
# have overlapping streak episodes, which makes the "nearest prior start" pick
# only an approximation for those members — flag it rather than silently trust it
streak_check = streak.sort_values(["member_id", "streak_start_date"]).copy()
streak_check["prev_end"] = streak_check.groupby("member_id")["streak_end_date"].shift()
overlap_streaks = streak_check[streak_check["streak_start_date"] < streak_check["prev_end"]]

log_merge_check(
    "streak_overlap_caveat",
    "streak_history_episodes",
    passed_count=len(streak) - len(overlap_streaks),
    total_count=len(streak),
    note="overlapping streak episodes for the same member exist upstream — as-of join picks nearest "
         "start, which may not always be the semantically 'correct' streak for a handful of members"
)

[streaks_asof_join_row_conservation] merged: 0/206732 flagged (0.0%) — merge_asof is fan-out safe by construction — row count must equal engagement_events
[streak_match_rate] merged: 190659/206732 flagged (92.23%) — share of events that fall inside a known streak episode for that member
[streak_overlap_caveat] streak_history_episodes: 509/40177 flagged (1.27%) — overlapping streak episodes for the same member exist upstream — as-of join picks nearest start, which may not always be the semantically 'correct' streak for a handful of members


In [10]:
assert len(merged) == len(engagement_events), "Row count drifted from engagement_events — a join fanned out"
print(f"Final merged analytical table: {merged.shape}")
print(f"Row count matches engagement_events exactly: {len(merged) == len(engagement_events)}")

# member coverage sanity check
print(f"\nDistinct members represented in final table: {merged['member_id'].nunique()} / {members['member_id'].nunique()} total members")

validation_report = pd.DataFrame(merge_validation_log)
print("\n=== PR6 Merge Validation Report ===")
print(validation_report.to_string(index=False))

Final merged analytical table: (206732, 60)
Row count matches engagement_events exactly: True

Distinct members represented in final table: 16956 / 17500 total members

=== PR6 Merge Validation Report ===
                                   check                      dataset  failed_or_flagged  total_rows  pct_flagged                                                                                                                                                                                note
            union_row_count_conservation            engagement_events                  0      206732         0.00                                                                                                         rows after union must exactly equal checkin + app row counts (no gain/loss)
              source_event_id_uniqueness            engagement_events                 10      206732         0.00                                                                 raw source_event_id not guara

In [12]:
engagement_events.to_csv(INTERIM_DIR / "engagement_events_unified.csv", index=False)
merged.to_csv(PROCESSED_DIR / "streakforge_merged.csv", index=False)
validation_report.to_csv(INTERIM_DIR / "pr6_merge_validation_report.csv", index=False)

print("Saved:")
print(f" - {INTERIM_DIR / 'engagement_events_unified.csv'}  ({len(engagement_events)} rows)")
print(f" - {PROCESSED_DIR / 'streakforge_merged.csv'}        ({len(merged)} rows, {merged.shape[1]} columns)")
print(f" - {INTERIM_DIR / 'pr6_merge_validation_report.csv'} ({len(validation_report)} checks logged)")

Saved:
 - ..\data\interim\engagement_events_unified.csv  (206732 rows)
 - ..\data\processed\streakforge_merged.csv        (206732 rows, 60 columns)
 - ..\data\interim\pr6_merge_validation_report.csv (11 checks logged)
